In [1]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.path.abspath(".."))

In [ ]:
%load_ext autoreload
%autoreload 2

import foldingdiff.modelling as modelling
import foldingdiff.datasets as dsets
from torch.utils.data import DataLoader
# from huggingface_hub import snapshot_download

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
dirpath = "../data/dompdb"
len(os.listdir(dirpath))
os.listdir(dirpath)[:5]

['12asA00', '152lA00', '153lA00', '155cA00', '16pkA02']

In [4]:
clean_dset = dsets.CathCanonicalAnglesOnlyDataset(pad=128, trim_strategy='randomcrop')

In [6]:
noised_dset = dsets.NoisedAnglesDataset(clean_dset, timesteps=1000, beta_schedule='cosine')

In [ ]:
dl = DataLoader(noised_dset, batch_size=32, shuffle=False)

In [20]:
x = next(iter(dl))
m = modelling.BertForDiffusion.from_dir("../config_jsons/")
predicted_noise = m(x['corrupted'], x['t'], x['attn_mask'])

Using time embedding: GaussianFourierProjection()
Using loss: [functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x0000028BB35BE430>, beta=0.3141592653589793)]


In [25]:
type(x)
x.keys()

dict_keys(['angles', 'coords', 'attn_mask', 'position_ids', 'lengths', 'corrupted', 't', 'known_noise', 'sqrt_alphas_cumprod_t', 'sqrt_one_minus_alphas_cumprod_t'])

In [21]:
predicted_noise

tensor([[[-0.7217, -0.7516,  0.1253, -0.3645, -1.2591,  0.5084],
         [-0.1577,  0.2862, -0.2626, -0.7626, -0.2822, -1.1752],
         [-1.5956, -0.1071, -0.5443,  0.6597, -1.0686,  0.1279],
         ...,
         [ 0.1233,  0.0680,  0.4835,  1.0055, -0.6249, -0.6854],
         [-1.0890, -0.1082,  0.3250,  0.9377, -0.3994,  1.0009],
         [-1.5143,  0.4151,  0.3157, -0.2950, -0.6164,  0.3076]],

        [[ 0.5541, -0.3036, -0.1874,  0.8849, -1.6522,  0.2970],
         [ 0.4541,  0.1263, -0.1889, -0.4067,  1.5651,  1.4124],
         [-0.6680,  0.4914,  1.3530, -1.3586, -0.8005,  0.8816],
         ...,
         [-0.8610, -0.5055, -2.0604, -0.1068, -0.7067,  0.6252],
         [-0.6235,  0.1739,  1.4153, -2.6367, -1.8716, -0.3002],
         [ 0.5767, -0.8967, -1.0156, -1.1257,  0.2197, -0.3807]],

        [[ 0.6345,  0.4788, -0.8540,  0.0556, -1.4229, -0.8246],
         [-1.0115, -0.1199,  1.9598, -1.7048,  0.2952, -1.5548],
         [-0.7663, -0.5191,  0.4683,  1.0598, -1.0523,  0.

In [22]:
import torch
import torch.nn as nn